<a href="https://colab.research.google.com/github/MuizSarwar/Deep-Learning-study-materials-/blob/main/Pytorch_training_pipe(deep_learning_cls_03).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Libraries
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder

In [3]:
#dataset load
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
#drop columns
df.drop(columns=['id','Unnamed: 32'],inplace=True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [5]:
#Target & Features
X = df.drop(['diagnosis'],axis=1) #Features
y = df['diagnosis'] #Target

#Train test split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

In [6]:
#Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
#Label Encoding on Target
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [8]:
#Numpy array to Pytorch tensor :
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [9]:
#Define the model:
class SimpleNN():
  def __init__(self,X) -> None:
    self.weights = torch.rand(X.shape[1],1,dtype=torch.float64,requires_grad=True)
    self.bias = torch.zeros(1,dtype=torch.float64,requires_grad=True)


  def forward(self,X):
    z = torch.matmul(X,self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self,y_pred,y):
    epsilon = 1e-8
    y_pred = torch.clamp(y_pred,epsilon,1-epsilon)
    loss = -(y * torch.log(y_pred) + (1-y) * torch.log(1-y_pred)).mean()
    return loss


In [10]:
#parameters
learning_rate = 0.1
epochs = 25

In [14]:
#Training Pipeline
model = SimpleNN(X_train_tensor)

for epoch in range(epochs):
  #forward pass
  y_pred = model.forward(X_train_tensor)

  #loss
  loss = model.loss_function(y_pred,y_train_tensor)

  #backward
  loss.backward()

  #parameters update
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  #zero gradient
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss}')


Epoch: 1, Loss: 3.9798798409244984
Epoch: 2, Loss: 3.8462989653793476
Epoch: 3, Loss: 3.7041377358723886
Epoch: 4, Loss: 3.55664315056436
Epoch: 5, Loss: 3.408869495270721
Epoch: 6, Loss: 3.254802320364387
Epoch: 7, Loss: 3.097028762341393
Epoch: 8, Loss: 2.9320696566431526
Epoch: 9, Loss: 2.7672534265822466
Epoch: 10, Loss: 2.601220387895882
Epoch: 11, Loss: 2.436729373333355
Epoch: 12, Loss: 2.2753840768739537
Epoch: 13, Loss: 2.115978427996967
Epoch: 14, Loss: 1.9592645501308437
Epoch: 15, Loss: 1.8066705668518221
Epoch: 16, Loss: 1.6616405037711268
Epoch: 17, Loss: 1.523324288259545
Epoch: 18, Loss: 1.3947517321100602
Epoch: 19, Loss: 1.280245965930573
Epoch: 20, Loss: 1.180809465895188
Epoch: 21, Loss: 1.0967437757955336
Epoch: 22, Loss: 1.0275328563787607
Epoch: 23, Loss: 0.9718753444055068
Epoch: 24, Loss: 0.927853153429895
Epoch: 25, Loss: 0.8932354587279847


In [20]:
#Evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.75).float()
  accuracy = (y_pred == y_test_tensor).float().mean()  #Accuracy = (TP+TN)/(TP+TN+FP+FN)
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5820252299308777
